In [4]:
import cv2
import torch
import os
import sys
import time
from datetime import datetime
import math
import gc # For garbage collection
import yaml

# --- Add project root to Python path ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project Root: {project_root}")

from models.model_loader import load_model
from core.video_processor import VideoProcessor
from video_reader import PyVideoReader

MODEL_PATH = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_dynamic.engine"
DATA_YAML_PATH = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract.yaml"
MODEL_INPUT_SIZE = 640
CONF_THRESHOLD = 0.05
IOU_THRESHOLD = 0.6
PYVIDEOREADER_DECODE_SEGMENT_SIZE = 50

# --- Configuration for this Benchmark Run ---
VIDEO_PATH = r"C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4"
# VIDEO_PATH = r"C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 minute test - 4 Way Intersection.mp4"
OUTPUT_DIR_BASE = r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs"

# --- Configuration for this Benchmark Run ---
FRAME_INTERVAL_CONFIG = 25
CONF_THRESHOLD_OVERRIDE = None
MAX_FRAMES_TO_PROCESS = 5
timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")
video_name_stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem}_{timestamp_run}")

vp = VideoProcessor('hardcoded',DATA_YAML_PATH=DATA_YAML_PATH)
effective_conf_thresh = CONF_THRESHOLD_OVERRIDE if CONF_THRESHOLD_OVERRIDE is not None else CONF_THRESHOLD
model = load_model(MODEL_PATH, vp.device, effective_conf_thresh, IOU_THRESHOLD) # load_model takes device

with open(DATA_YAML_PATH, 'r', encoding='utf-8') as f: # Added encoding
    data_yaml = yaml.safe_load(f)

num_classes = data_yaml['nc']
class_names_map = vp.class_id_to_name_map
# class_names_map = {}
default_class_label = ''

Project Root: c:\Users\swest\Desktop\traffic-capturing\traffic-reporting
Using GPU: NVIDIA GeForce RTX 3090


Benchmark - Standard Processing

In [2]:
print(f"\n--- Starting Benchmark Processing ---")
os.makedirs(OUTPUT_DIR_RUN, exist_ok=True)
print(f"Annotated frames will be saved to: {OUTPUT_DIR_RUN}")
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
if _cap_cv_check.isOpened():
    total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
    original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
    _cap_cv_check.release()
else:
    print(f"Error: OpenCV could not open video {VIDEO_PATH} to get properties.")
    raise IOError(f"Failed to get video properties for {VIDEO_PATH}")

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

frames_actually_processed = 0
total_preprocessing_time_ms = 0
total_inference_time_ms = 0
raw_frame_index_overall = 0

cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

current_batch_for_preprocessing = []

pvr_instance = None
print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
pvr_instance = PyVideoReader(VIDEO_PATH, device='cpu',
                                resize_shorter_side=MODEL_INPUT_SIZE,
                                resize_longer_side=MODEL_INPUT_SIZE)

num_segments = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
print(f"PVR: Segments={num_segments}")

for seg_idx in range(num_segments):
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
    segment_start_raw_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
    segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
    if segment_start_raw_idx >= segment_end_raw_idx: continue
    
    segment_frames_np_rgb = pvr_instance.decode(start_frame=segment_start_raw_idx, end_frame=segment_end_raw_idx)
    if segment_frames_np_rgb is None or segment_frames_np_rgb.shape[0] == 0: continue

    for frame_in_segment_idx in range(segment_frames_np_rgb.shape[0]):
        current_original_video_frame_idx = segment_start_raw_idx + frame_in_segment_idx
        raw_frame_index_overall = current_original_video_frame_idx

        if current_original_video_frame_idx % frame_skip_interval != 0: continue

        original_frame_bgr = None
        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
        ret_orig, original_frame_bgr = cap_for_originals.read()
        if not ret_orig: continue

        frames_actually_processed += 1
        print(f"PVR Processing Original Frame #{current_original_video_frame_idx} (Actual Proc. #{frames_actually_processed})")
        
        frame_content_for_model = segment_frames_np_rgb[frame_in_segment_idx]

        metadata = {'is_rgb': True, 'source': 'pvr_benchmark'}
        current_batch_for_preprocessing.append((frame_content_for_model, metadata))
        
        start_preprocess_t = time.monotonic()
        preprocessed_tensor = vp._preprocess_batch(current_batch_for_preprocessing)
        current_batch_for_preprocessing.clear() # Clear after use
        preprocessing_t_ms = (time.monotonic() - start_preprocess_t) * 1000
        total_preprocessing_time_ms += preprocessing_t_ms
        print(f"  Preprocessing time (via vp._preprocess_batch): {preprocessing_t_ms:.2f} ms")

        if preprocessed_tensor is None:
            print("  Skipping frame: Preprocessing returned None.")
            continue

        # Inference (same as before)
        start_inference_t = time.monotonic()
        with torch.no_grad(): results = model(preprocessed_tensor, verbose=False)
        inference_t_ms = (time.monotonic() - start_inference_t) * 1000
        total_inference_time_ms += inference_t_ms
        print(f"  Inference time: {inference_t_ms:.2f} ms")

        # Postprocessing & Drawing (same as before, on original_frame_bgr)
        annotated_display_frame = original_frame_bgr.copy()
        result_obj = results[0]
        num_detections_on_frame = 0
        if hasattr(result_obj, 'boxes') and result_obj.boxes is not None:
            for box_data in result_obj.boxes.data:
                box_cpu = box_data.cpu().numpy(); x1m,y1m,x2m,y2m,conf,cls_id_f = box_cpu
                cls_id = int(cls_id_f); num_detections_on_frame +=1
                label_str = class_names_map.get(cls_id, f"{default_class_label}_{cls_id}")
                oh, ow = annotated_display_frame.shape[:2]
                x1,y1,x2,y2 = int(x1m*ow/MODEL_INPUT_SIZE),int(y1m*oh/MODEL_INPUT_SIZE),int(x2m*ow/MODEL_INPUT_SIZE),int(y2m*oh/MODEL_INPUT_SIZE)
                cv2.rectangle(annotated_display_frame,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.putText(annotated_display_frame,f"{label_str}:{conf:.2f}",(x1,y1-10 if y1>20 else y1+20),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
            print(f"  Detections: {num_detections_on_frame}")
        else: print("  No detections.")

        cv2.imwrite(os.path.join(OUTPUT_DIR_RUN, f"frame_{current_original_video_frame_idx:06d}_pvr.jpg"), annotated_display_frame)
        
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break
    del segment_frames_np_rgb; gc.collect()
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS: break


if cap_for_originals.isOpened(): cap_for_originals.release()
gc.collect()

# --- Final Summary (same as before) ---
print("\n--- Benchmark Run Summary ---")
if frames_actually_processed > 0:
    avg_preprocess_ms = total_preprocessing_time_ms / frames_actually_processed
    avg_inference_ms = total_inference_time_ms / frames_actually_processed
    print(f"Total Video Frames (approx): {total_video_frames}")
    print(f"Frames Processed: {frames_actually_processed}")
    print(f"Avg Preprocessing Time: {avg_preprocess_ms:.2f} ms/frame")
    print(f"Avg Inference Time: {avg_inference_ms:.2f} ms/frame")
    total_avg_ms = avg_preprocess_ms + avg_inference_ms
    print(f"Avg Total (Enh+PreP+Inf): {total_avg_ms:.2f} ms/frame")
    if total_avg_ms > 0 : print(f"Equivalent FPS (processed): {1000 / total_avg_ms:.2f} FPS")
else: print("No frames processed.")
if os.path.exists(OUTPUT_DIR_RUN) and len(os.listdir(OUTPUT_DIR_RUN)) > 0 :
    print(f"Annotated frames saved to: {OUTPUT_DIR_RUN}")
else: print(f"SAVE_ANNOTATED_FRAMES was True, but no frames saved to {OUTPUT_DIR_RUN}.")
print("Notebook execution finished.")


--- Starting Benchmark Processing ---
Annotated frames will be saved to: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 minute test - 4 Way Intersection_20250622_2310
Video Properties: ~7545 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 minute test - 4 Way Intersection.mp4, target: 640x640
PVR: Segments=151
PVR Processing Original Frame #0 (Actual Proc. #1)
  Preprocessing time (via vp._preprocess_batch): 32.00 ms
Loading C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\fixed_cupola_6120_640px_dynamic.engine for TensorRT inference...


WARNING Metadata not found for 'model=C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\fixed_cupola_6120_640px_dynamic.engine'
  Inference time: 1015.00 ms
  Detections: 4
PVR Processing Original Frame #25 (Actual Proc. #2)
  Preprocessing time (via vp._preprocess_batch): 0.00 ms
  Inference time: 16.00 ms
  Detections: 3
PVR Processing Original Frame #50 (Actual Proc. #3)
  Preprocessing time (via vp._preprocess_batch): 0.00 ms
  Inference time: 16.00 ms
  Detections: 4
PVR Processing Original Frame #75 (Actual Proc. #4)
  Preprocessing time (via vp._preprocess_batch): 0.00 ms
  Inference time: 0.00 ms
  Detections: 5
PVR Processing Original Frame #100 (Actual Proc. #5)
  Preprocessing time (via vp._preprocess_batch): 0.00 ms
  Inference time: 0.00 ms
  Detections: 4

--- Benchmark Run Summary ---
Total Video Frames (approx): 7545
Frames Processed: 5
Avg Preprocessing Time: 6.40 ms/frame
Avg Inference Time: 209.40 ms/frame
Avg Total (Enh+PreP+Inf): 215.80 ms/f

----------------------------------

-----------------------------------

Using patched_yolo_infer

In [2]:
def extract_raw_detections_from_crops(element_crops, conf_threshold=0.08):
    """Extract raw detections from CropElement objects and transform to original image coordinates"""
    raw_boxes = []
    raw_confidences = []
    raw_classes = []
    
    print(f"    DEBUG: Extracting from {len(element_crops.crops)} crops...")
    
    # Get scaling factors to map from resized image back to original
    original_image = element_crops.crops[0].source_image if element_crops.crops else None
    resized_image = element_crops.crops[0].source_image_resized if element_crops.crops else None
    
    scale_factor_x = 1.0
    scale_factor_y = 1.0
    
    if original_image is not None and resized_image is not None:
        scale_factor_x = original_image.shape[1] / resized_image.shape[1]  # width scaling
        scale_factor_y = original_image.shape[0] / resized_image.shape[0]  # height scaling
        print(f"    DEBUG: Scale factors - X: {scale_factor_x:.3f}, Y: {scale_factor_y:.3f}")
    
    crops_with_detections = 0
    total_detections_found = 0
    
    for crop_idx, crop in enumerate(element_crops.crops):
        # Check if crop has detections - fix the boolean array issue
        has_detections = (hasattr(crop, 'detected_xyxy') and 
                         crop.detected_xyxy is not None and 
                         len(crop.detected_xyxy) > 0)
        
        has_confidences = (hasattr(crop, 'detected_conf') and 
                          crop.detected_conf is not None and 
                          len(crop.detected_conf) > 0)
        
        has_classes = (hasattr(crop, 'detected_cls') and 
                      crop.detected_cls is not None and 
                      len(crop.detected_cls) > 0)
        
        if not (has_detections and has_confidences and has_classes):
            continue
        
        crops_with_detections += 1
        crop_detections = len(crop.detected_xyxy)
        total_detections_found += crop_detections
        
        print(f"    DEBUG: Crop {crop_idx} has {crop_detections} detections")
        
        # Process each detection in this crop
        for det_idx in range(crop_detections):
            try:
                conf = float(crop.detected_conf[det_idx])
                if conf < conf_threshold:
                    continue
                    
                # Get crop-relative coordinates
                x1_crop, y1_crop, x2_crop, y2_crop = crop.detected_xyxy[det_idx]
                cls_id = int(crop.detected_cls[det_idx])
                
                # Get crop position in resized image
                crop_x_start = crop.x_start
                crop_y_start = crop.y_start
                
                # Transform to resized image coordinates
                x1_resized = x1_crop + crop_x_start
                y1_resized = y1_crop + crop_y_start
                x2_resized = x2_crop + crop_x_start
                y2_resized = y2_crop + crop_y_start
                
                # Transform to original image coordinates
                x1_orig = x1_resized * scale_factor_x
                y1_orig = y1_resized * scale_factor_y
                x2_orig = x2_resized * scale_factor_x
                y2_orig = y2_resized * scale_factor_y
                
                raw_boxes.append([x1_orig, y1_orig, x2_orig, y2_orig])
                raw_confidences.append(conf)
                raw_classes.append(cls_id)
                
            except Exception as e:
                print(f"    DEBUG: Error processing detection {det_idx} in crop {crop_idx}: {e}")
                continue
    
    print(f"    DEBUG: {crops_with_detections}/{len(element_crops.crops)} crops had detections")
    print(f"    DEBUG: Total detections found: {total_detections_found}, above threshold: {len(raw_boxes)}")
    
    return raw_boxes, raw_confidences, raw_classes

def annotate_detections(frame, boxes, confidences, classes, class_map, color, postprocess_conf_threshold, prefix=""):
    """Annotate frame with detections"""
    annotated_frame = frame.copy()
    num_detections = 0
    
    for i in range(len(boxes)):
        cls_id = classes[i]
        if cls_id>num_classes-1:
            continue
        conf = confidences[i]
        if conf < postprocess_conf_threshold:
            continue
            
        num_detections += 1
        x1, y1, x2, y2 = map(int, boxes[i])
        
        # Ensure coordinates are within frame bounds
        frame_h, frame_w = frame.shape[:2]
        x1 = max(0, min(x1, frame_w-1))
        y1 = max(0, min(y1, frame_h-1))
        x2 = max(0, min(x2, frame_w-1))
        y2 = max(0, min(y2, frame_h-1))
        
        if x2 <= x1 or y2 <= y1:
            continue
        
        label_str = class_map.get(cls_id, str(cls_id))
        
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 1)
        cv2.putText(annotated_frame, f"{prefix}{label_str}:{conf:.2f}", 
                   (x1, y1-10 if y1>20 else y1+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    print("annotation complete")
    return annotated_frame, num_detections

In [6]:
# Import the patched_yolo_infer library
from patched_yolo_infer import MakeCropsDetectThem, CombineDetections
MAX_FRAMES_TO_PROCESS = 5
CROP_HEIGHT = 640  
CROP_WIDTH = 640   
OVERLAP_HEIGHT_PERCENTAGE = 15  
OVERLAP_WIDTH_PERCENTAGE = 15   
POSTPROCESS_CONF_THRESHOLD = 0.15
BASE_MODEL_CONF_FOR_CROPS = 0.08
POSTPROCESS_IOU_THRESHOLD = 0.8
NMS_THRESHOLD = 0.85
timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")

print(f"  Crop Params: Crop HxW={CROP_HEIGHT}x{CROP_WIDTH}, Overlap={OVERLAP_HEIGHT_PERCENTAGE}%, Post-Conf={POSTPROCESS_CONF_THRESHOLD}")

# Base Model Confidence for individual crops
if CONF_THRESHOLD_OVERRIDE is None or CONF_THRESHOLD_OVERRIDE > BASE_MODEL_CONF_FOR_CROPS:
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_CROPS
    print(f"  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to {CONF_THRESHOLD_OVERRIDE}")

# Advanced crop inference settings
USE_BATCH_INFERENCE = True  
USE_MEMORY_OPTIMIZE = False  
ENABLE_PROGRESS_TRACKING = False  
CLASS_AGNOSTIC_NMS = False  
INTELLIGENT_SORTER = True  
SORTER_BINS = 7  
MATCH_METRIC = 'IOS'
class_names_map = {}

# Enhanced output directory structure
crop_str_override = "_crops"

if os.path.exists(VIDEO_PATH):
    video_name_stem_override = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
else:
    video_name_stem_override = "video_not_found"

# Create separate directories for NMS and raw detections
OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem_override}{crop_str_override}_{timestamp_run}")
OUTPUT_DIR_NMS = os.path.join(OUTPUT_DIR_RUN, "nms_filtered")
OUTPUT_DIR_RAW = os.path.join(OUTPUT_DIR_RUN, "raw_detections")

os.makedirs(OUTPUT_DIR_NMS, exist_ok=True)
os.makedirs(OUTPUT_DIR_RAW, exist_ok=True)
print(f"  NMS Output: {OUTPUT_DIR_NMS}")
print(f"  Raw Output: {OUTPUT_DIR_RAW}")

print("Crop-based inference setup applied.")
print("-" * 50)

print(f"\n--- Starting Benchmark Processing ---")

# Get Video Properties
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)
_cap_cv_check.release()

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")
frame_skip_interval = FRAME_INTERVAL_CONFIG

# Initialize Enhanced Counters
frames_actually_processed = 0
total_preprocessing_time_ms_std = 0
total_inference_time_ms = 0
total_enhancement_time_ms = 0
total_nms_combination_time_ms = 0
total_raw_processing_time_ms = 0
raw_frame_index_overall = 0
total_crops_generated = 0
total_raw_detections = 0
total_nms_detections = 0

# OpenCV VideoCapture for getting original frames for annotation
cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    print(f"CRITICAL ERROR: OpenCV could not open video {VIDEO_PATH} for original frames.")
    raise IOError(f"Failed to open video for original frames: {VIDEO_PATH}")

# Video Reading Loop Setup
reader_frame_source = None
segment_iterator = range(1)
pvr_segment_details = {}

print(f"PVR: Initializing for {VIDEO_PATH}, target: {MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}")
reader_frame_source = PyVideoReader(VIDEO_PATH, device='cpu',
                                    resize_shorter_side=MODEL_INPUT_SIZE,
                                    resize_longer_side=MODEL_INPUT_SIZE)
pvr_segment_details['num_segments'] = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE)
segment_iterator = range(pvr_segment_details['num_segments'])
print(f"PVR Source: Segments={pvr_segment_details['num_segments']}")

# Main Loop
for seg_idx in segment_iterator:
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
        break

    current_pvr_segment_frames_rgb = None
    pvr_current_segment_start_idx = 0

    pvr_current_segment_start_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
    segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
    if pvr_current_segment_start_idx >= segment_end_raw_idx:
        continue
    
    print(f"\nPVR: Decoding segment {seg_idx+1}/{pvr_segment_details['num_segments']} "
            f"(raw frames {pvr_current_segment_start_idx} to {segment_end_raw_idx-1})")
    current_pvr_segment_frames_rgb = reader_frame_source.decode(start_frame=pvr_current_segment_start_idx,
                                                                end_frame=segment_end_raw_idx)
    if current_pvr_segment_frames_rgb is None or current_pvr_segment_frames_rgb.shape[0] == 0:
        print(f"PVR: Segment {seg_idx+1} returned no frames.")
        continue
    print(f"PVR: Decoded {current_pvr_segment_frames_rgb.shape[0]} frames from segment.")
    
    frame_idx_in_current_pvr_segment = -1

    while True:
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break
        
        current_original_video_frame_idx = -1
        is_input_rgb = False

        frame_idx_in_current_pvr_segment += 1
        if frame_idx_in_current_pvr_segment >= current_pvr_segment_frames_rgb.shape[0]:
            break
        current_original_video_frame_idx = pvr_current_segment_start_idx + frame_idx_in_current_pvr_segment
        is_input_rgb = True

        raw_frame_index_overall = current_original_video_frame_idx

        if current_original_video_frame_idx % frame_skip_interval != 0:
            continue

        # Get original BGR frame
        original_frame_bgr = None
        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_video_frame_idx)
        ret_orig, original_frame_bgr = cap_for_originals.read()
        if not ret_orig or original_frame_bgr is None:
            print(f"Warning: OpenCV could not read original frame at index {current_original_video_frame_idx} for drawing.")
            continue

        frames_actually_processed += 1
        print(f"Processing Original Frame #{current_original_video_frame_idx} (Actual Processed Frame #{frames_actually_processed})")

        image_to_infer_on_bgr = original_frame_bgr.copy()
        
        # Inference Block - Crop-based inference
        current_inf_block_start_time = time.monotonic()
        print(f"  Running crop-based prediction...")
        
        try:
            # Create crops and run inference
            element_crops = MakeCropsDetectThem(
                image=image_to_infer_on_bgr,
                model=model,
                imgsz=MODEL_INPUT_SIZE,
                conf=BASE_MODEL_CONF_FOR_CROPS,
                iou=POSTPROCESS_IOU_THRESHOLD,
                classes_list=None,
                segment=False,
                shape_x=CROP_WIDTH,
                shape_y=CROP_HEIGHT,
                overlap_x=OVERLAP_WIDTH_PERCENTAGE,
                overlap_y=OVERLAP_HEIGHT_PERCENTAGE,
                show_crops=False,
                show_processing_status=ENABLE_PROGRESS_TRACKING,
                resize_initial_size=True,
                memory_optimize=USE_MEMORY_OPTIMIZE,
                batch_inference=USE_BATCH_INFERENCE
            )
            
            num_crops = len(element_crops.crops)
            total_crops_generated += num_crops
            print(f"    Generated {num_crops} crops from frame")
            
            # Extract raw detections from CropElement objects
            raw_processing_start = time.monotonic()
            raw_boxes, raw_confidences, raw_classes = extract_raw_detections_from_crops(element_crops, BASE_MODEL_CONF_FOR_CROPS)
            raw_processing_time_ms = (time.monotonic() - raw_processing_start) * 1000
            total_raw_processing_time_ms += raw_processing_time_ms
            
            num_raw_detections = len(raw_boxes)
            total_raw_detections += num_raw_detections
            print(f"    Raw detections: {num_raw_detections}")
            
            # Combine detections using NMS
            nms_start_time = time.monotonic()
            result = CombineDetections(
                element_crops=element_crops,
                nms_threshold=NMS_THRESHOLD,
                match_metric=MATCH_METRIC,
                class_agnostic_nms=CLASS_AGNOSTIC_NMS,
                intelligent_sorter=INTELLIGENT_SORTER,
                sorter_bins=SORTER_BINS
            )
            nms_time_ms = (time.monotonic() - nms_start_time) * 1000
            total_nms_combination_time_ms += nms_time_ms
            
            # Count NMS filtered detections
            num_nms_detections = len(result.filtered_boxes) if hasattr(result, 'filtered_boxes') else 0
            total_nms_detections += num_nms_detections
            print(f"    NMS filtered detections: {num_nms_detections}")
            
        except Exception as e:
            print(f"  ERROR during crop-based inference: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        current_inf_block_time_ms = (time.monotonic() - current_inf_block_start_time) * 1000
        total_inference_time_ms += current_inf_block_time_ms
        print(f"  Inference times - Total: {current_inf_block_time_ms:.2f}ms, Raw: {raw_processing_time_ms:.2f}ms, NMS: {nms_time_ms:.2f}ms")

        # Raw detections (red color)
        raw_annotated_frame, num_raw_drawn = annotate_detections(
            original_frame_bgr, raw_boxes, raw_confidences, raw_classes,
            class_names_map, (0, 0, 255), POSTPROCESS_CONF_THRESHOLD, ""
        )
        
        # NMS filtered detections (green color)
        nms_boxes = result.filtered_boxes if hasattr(result, 'filtered_boxes') else []
        nms_confidences = result.filtered_confidences if hasattr(result, 'filtered_confidences') else []
        nms_classes = result.filtered_classes_id if hasattr(result, 'filtered_classes_id') else []
        
        nms_annotated_frame, num_nms_drawn = annotate_detections(
            original_frame_bgr, nms_boxes, nms_confidences, nms_classes,
            class_names_map, (0, 255, 0), POSTPROCESS_CONF_THRESHOLD, ""
        )
        
        # Save both versions
        frame_filename = f"frame_{current_original_video_frame_idx:06d}.jpg"
        cv2.imwrite(os.path.join(OUTPUT_DIR_RAW, frame_filename), raw_annotated_frame)
        cv2.imwrite(os.path.join(OUTPUT_DIR_NMS, frame_filename), nms_annotated_frame)
        
        print(f"  Saved - Raw: {num_raw_drawn} detections, NMS: {num_nms_drawn} detections")
        
        # Memory cleanup
        del element_crops, result, raw_annotated_frame, nms_annotated_frame
        gc.collect()
        
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed >= MAX_FRAMES_TO_PROCESS:
            break

    del current_pvr_segment_frames_rgb
    gc.collect()

# Release Resources
if reader_frame_source: 
    del reader_frame_source
    gc.collect()
elif not reader_frame_source and reader_frame_source.isOpened(): 
    reader_frame_source.release()
if cap_for_originals.isOpened(): 
    cap_for_originals.release()
gc.collect()

# Enhanced Final Summary
print("\n--- Enhanced Benchmark Summary ---")
if frames_actually_processed > 0:
    avg_inference_block_ms = total_inference_time_ms / frames_actually_processed
    avg_nms_ms = total_nms_combination_time_ms / frames_actually_processed
    avg_raw_processing_ms = total_raw_processing_time_ms / frames_actually_processed
    avg_crops_per_frame = total_crops_generated / frames_actually_processed
    avg_raw_detections_per_frame = total_raw_detections / frames_actually_processed
    avg_nms_detections_per_frame = total_nms_detections / frames_actually_processed
    
    print(f"**Processing Statistics:**")
    print(f"  Total Video Frames: {total_video_frames}")
    print(f"  Frames Processed: {frames_actually_processed}")
    print(f"  Total Crops Generated: {total_crops_generated} (avg: {avg_crops_per_frame:.1f}/frame)")
    print(f"  Total Raw Detections: {total_raw_detections} (avg: {avg_raw_detections_per_frame:.1f}/frame)")
    print(f"  Total NMS Detections: {total_nms_detections} (avg: {avg_nms_detections_per_frame:.1f}/frame)")
    if total_raw_detections > 0:
        print(f"  NMS Reduction Rate: {((total_raw_detections - total_nms_detections) / total_raw_detections) * 100:.1f}%")
    
    print(f"\n**Timing Performance:**")
    print(f"  Raw Processing: {avg_raw_processing_ms:.2f} ms/frame")
    print(f"  NMS Combination: {avg_nms_ms:.2f} ms/frame")
    print(f"  Total Inference Block: {avg_inference_block_ms:.2f} ms/frame")
    
    total_avg_ms_cycle = avg_inference_block_ms
    print(f"  Total Cycle Time: {total_avg_ms_cycle:.2f} ms/frame")
    
    if total_avg_ms_cycle > 0: 
        equivalent_fps = 1000 / total_avg_ms_cycle
        print(f"  Equivalent FPS: {equivalent_fps:.2f}")
        
        # Performance breakdown
        print(f"\n**Performance Breakdown:**")
        print(f"  Raw Processing: {(avg_raw_processing_ms/total_avg_ms_cycle)*100:.1f}%")
        print(f"  NMS Combination: {(avg_nms_ms/total_avg_ms_cycle)*100:.1f}%")
else:
    print("No frames were processed.")

raw_files = len(os.listdir(OUTPUT_DIR_RAW)) if os.path.exists(OUTPUT_DIR_RAW) else 0
nms_files = len(os.listdir(OUTPUT_DIR_NMS)) if os.path.exists(OUTPUT_DIR_NMS) else 0
print(f"\n**Output Files:**")
print(f"  Raw detections: {raw_files} files in {OUTPUT_DIR_RAW}")
print(f"  NMS filtered: {nms_files} files in {OUTPUT_DIR_NMS}")

  Crop Params: Crop HxW=640x640, Overlap=15%, Post-Conf=0.15
  NMS Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250622_1500\nms_filtered
  Raw Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250622_1500\raw_detections
Crop-based inference setup applied.
--------------------------------------------------

--- Starting Benchmark Processing ---
Video Properties: ~7500 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4, target: 640x640
PVR Source: Segments=150

PVR: Decoding segment 1/150 (raw frames 0 to 49)
PVR: Decoded 50 frames from segment.
Processing Original Frame #0 (Actual Processed Frame #1)
  Running crop-based prediction...
  ERROR during crop-based inference: input size torch.Size([8, 3, 640, 640]) not equal 

Traceback (most recent call last):
  File "C:\Users\swest\AppData\Local\Temp\ipykernel_16528\637979833.py", line 157, in <module>
    element_crops = MakeCropsDetectThem(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 132, in __init__
    self._detect_objects_batch()
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 301, in _detect_objects_batch
    self._calculate_batch_inference(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 337, in _calculate_batch_inference
    predictions = model.predict(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\ultralytics\engine\model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "c:\Users\swest\.conda\e


PVR: Decoding segment 3/150 (raw frames 100 to 149)
PVR: Decoded 50 frames from segment.
Processing Original Frame #100 (Actual Processed Frame #2)
  Running crop-based prediction...
  ERROR during crop-based inference: input size torch.Size([8, 3, 640, 640]) not equal to max model size (64, 3, 640, 640)

PVR: Decoding segment 4/150 (raw frames 150 to 199)


Traceback (most recent call last):
  File "C:\Users\swest\AppData\Local\Temp\ipykernel_16528\637979833.py", line 157, in <module>
    element_crops = MakeCropsDetectThem(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 132, in __init__
    self._detect_objects_batch()
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 301, in _detect_objects_batch
    self._calculate_batch_inference(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 337, in _calculate_batch_inference
    predictions = model.predict(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\ultralytics\engine\model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "c:\Users\swest\.conda\e

PVR: Decoded 50 frames from segment.

PVR: Decoding segment 5/150 (raw frames 200 to 249)
PVR: Decoded 50 frames from segment.
Processing Original Frame #200 (Actual Processed Frame #3)
  Running crop-based prediction...
  ERROR during crop-based inference: input size torch.Size([8, 3, 640, 640]) not equal to max model size (64, 3, 640, 640)

PVR: Decoding segment 6/150 (raw frames 250 to 299)
PVR: Decoded 50 frames from segment.


Traceback (most recent call last):
  File "C:\Users\swest\AppData\Local\Temp\ipykernel_16528\637979833.py", line 157, in <module>
    element_crops = MakeCropsDetectThem(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 132, in __init__
    self._detect_objects_batch()
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 301, in _detect_objects_batch
    self._calculate_batch_inference(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 337, in _calculate_batch_inference
    predictions = model.predict(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\ultralytics\engine\model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "c:\Users\swest\.conda\e


PVR: Decoding segment 7/150 (raw frames 300 to 349)
PVR: Decoded 50 frames from segment.
Processing Original Frame #300 (Actual Processed Frame #4)
  Running crop-based prediction...
  ERROR during crop-based inference: input size torch.Size([8, 3, 640, 640]) not equal to max model size (64, 3, 640, 640)

PVR: Decoding segment 8/150 (raw frames 350 to 399)


Traceback (most recent call last):
  File "C:\Users\swest\AppData\Local\Temp\ipykernel_16528\637979833.py", line 157, in <module>
    element_crops = MakeCropsDetectThem(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 132, in __init__
    self._detect_objects_batch()
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 301, in _detect_objects_batch
    self._calculate_batch_inference(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 337, in _calculate_batch_inference
    predictions = model.predict(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\ultralytics\engine\model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "c:\Users\swest\.conda\e

PVR: Decoded 50 frames from segment.

PVR: Decoding segment 9/150 (raw frames 400 to 449)
PVR: Decoded 50 frames from segment.
Processing Original Frame #400 (Actual Processed Frame #5)
  Running crop-based prediction...
  ERROR during crop-based inference: input size torch.Size([8, 3, 640, 640]) not equal to max model size (64, 3, 640, 640)


Traceback (most recent call last):
  File "C:\Users\swest\AppData\Local\Temp\ipykernel_16528\637979833.py", line 157, in <module>
    element_crops = MakeCropsDetectThem(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 132, in __init__
    self._detect_objects_batch()
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 301, in _detect_objects_batch
    self._calculate_batch_inference(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\patched_yolo_infer\nodes\MakeCropsDetectThem.py", line 337, in _calculate_batch_inference
    predictions = model.predict(
  File "c:\Users\swest\.conda\envs\traffic_env_py310\lib\site-packages\ultralytics\engine\model.py", line 552, in predict
    return self.predictor.predict_cli(source=source) if is_cli else self.predictor(source=source, stream=stream)
  File "c:\Users\swest\.conda\e


--- Enhanced Benchmark Summary ---
**Processing Statistics:**
  Total Video Frames: 7500
  Frames Processed: 5
  Total Crops Generated: 0 (avg: 0.0/frame)
  Total Raw Detections: 0 (avg: 0.0/frame)
  Total NMS Detections: 0 (avg: 0.0/frame)

**Timing Performance:**
  Raw Processing: 0.00 ms/frame
  NMS Combination: 0.00 ms/frame
  Total Inference Block: 0.00 ms/frame
  Total Cycle Time: 0.00 ms/frame

**Output Files:**
  Raw detections: 0 files in C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250622_1500\raw_detections
  NMS filtered: 0 files in C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_20250622_1500\nms_filtered


----------------------------------

------------------------

Using custom modified batched patched_yolo_infer

In [5]:
import cv2
import numpy as np
from ultralytics import YOLO # Assuming this is how YOLO is imported
from collections import defaultdict
import math
import os
import time # For timing
import gc # For garbage collection
from datetime import datetime # For timestamp
from patched_yolo_infer_batched.nodes.MakeCropsDetectThem import MakeCropsDetectThem 
from patched_yolo_infer_batched.nodes.CombineDetections import CombineDetections, MCDTLike

FRAME_INTERVAL_CONFIG = 100
MAX_FRAMES_TO_PROCESS = 50
# class_names_map = {}

# --- Model Configuration ---
FIXED_PATCH_BATCH_SIZE_FOR_MODEL = 64 # How many patches to send to model.predict (will be padded)
CROP_HEIGHT = 640  
CROP_WIDTH = 640 
OVERLAP_HEIGHT_PERCENTAGE = 20  
OVERLAP_WIDTH_PERCENTAGE = 20   
POSTPROCESS_CONF_THRESHOLD = 0.3 # For final drawing of NMS results
BASE_MODEL_CONF_FOR_CROPS = 0.15 # Confidence for YOLO on individual crops
POSTPROCESS_IOU_THRESHOLD = 0.8
NMS_THRESHOLD = 0.85 # NMS IoU/IoS threshold for CombineDetections
timestamp_run = datetime.now().strftime("%Y%m%d_%H%M")


print(f"  Crop Params: Crop HxW={CROP_HEIGHT}x{CROP_WIDTH}, Overlap={OVERLAP_HEIGHT_PERCENTAGE}%, Post-Conf={POSTPROCESS_CONF_THRESHOLD}")

# Base Model Confidence for individual crops
if CONF_THRESHOLD_OVERRIDE is None or CONF_THRESHOLD_OVERRIDE > BASE_MODEL_CONF_FOR_CROPS:
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_CROPS
    print(f"  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to {CONF_THRESHOLD_OVERRIDE}")

# Advanced crop inference settings
CLASS_AGNOSTIC_NMS = False  
INTELLIGENT_SORTER = True  
SORTER_BINS = 7  
MATCH_METRIC = 'IOS'

print(f"Crop Params: Crop HxW={CROP_HEIGHT}x{CROP_WIDTH}, Overlap={OVERLAP_HEIGHT_PERCENTAGE}%, Post-Conf(Draw)={POSTPROCESS_CONF_THRESHOLD}")

# Base Model Confidence for individual crops
_conf_override_for_orchestrator = BASE_MODEL_CONF_FOR_CROPS # Default to this
if 'CONF_THRESHOLD_OVERRIDE' in globals() and CONF_THRESHOLD_OVERRIDE is not None:
    if CONF_THRESHOLD_OVERRIDE < BASE_MODEL_CONF_FOR_CROPS: # Use the stricter (lower) one for initial detection
        _conf_override_for_orchestrator = CONF_THRESHOLD_OVERRIDE
    print(f"  Effective CONF for base model on crops set to {_conf_override_for_orchestrator}")
else: # CONF_THRESHOLD_OVERRIDE not defined or None
    CONF_THRESHOLD_OVERRIDE = BASE_MODEL_CONF_FOR_CROPS # Ensure it's defined for later parts if needed
    print(f"  Effective CONF for base model on crops set to {_conf_override_for_orchestrator}")

ORCH_COMMON_MCDT_CONFIG = {
    'imgsz': MODEL_INPUT_SIZE, # Size for YOLO crop inference
    'conf': _conf_override_for_orchestrator, # Use the determined confidence
    'iou': POSTPROCESS_IOU_THRESHOLD, # IoU for YOLO crop inference
    'classes_list': None, # Or your specific class list
    'segment': False, # Set to True if using YOLO-Seg model
    'memory_optimize': False,
    'show_crops': False, # Usually False in benchmark
    'show_processing_status': False,
    'inference_extra_args': None
}

ORCH_CROP_PARAMS = {
    'shape_x': CROP_WIDTH,
    'shape_y': CROP_HEIGHT,
    'overlap_x': OVERLAP_WIDTH_PERCENTAGE,
    'overlap_y': OVERLAP_HEIGHT_PERCENTAGE,
}

# CombineDetections NMS settings
NMS_CONFIG = {
    'nms_threshold': NMS_THRESHOLD,
    'match_metric': MATCH_METRIC,
    'class_agnostic_nms': CLASS_AGNOSTIC_NMS,
    'intelligent_sorter': INTELLIGENT_SORTER,
    'sorter_bins': SORTER_BINS
}


crop_str_override = "_crops_ext_batch" # Indicate new method
video_name_stem_override = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

OUTPUT_DIR_RUN = os.path.join(OUTPUT_DIR_BASE, f"{video_name_stem_override}{crop_str_override}_{timestamp_run}")
OUTPUT_DIR_NMS = os.path.join(OUTPUT_DIR_RUN, "nms_filtered")
OUTPUT_DIR_RAW_CROPS = os.path.join(OUTPUT_DIR_RUN, "raw_crop_detections") # For per-crop raw before NMS

os.makedirs(OUTPUT_DIR_NMS, exist_ok=True)
os.makedirs(OUTPUT_DIR_RAW_CROPS, exist_ok=True)
print(f"NMS Output: {OUTPUT_DIR_NMS}")
print(f"Raw Crop Detections Output: {OUTPUT_DIR_RAW_CROPS}")
print("Crop-based inference setup applied.")
print("-" * 50)

print(f"\n--- Starting Benchmark Processing (Refactored) ---")

# Get Video Properties
total_video_frames = 0
original_video_fps = 0.0
_cap_cv_check = cv2.VideoCapture(VIDEO_PATH)
total_video_frames = int(_cap_cv_check.get(cv2.CAP_PROP_FRAME_COUNT))
original_video_fps = _cap_cv_check.get(cv2.CAP_PROP_FPS)

print(f"Video Properties: ~{total_video_frames} frames, {original_video_fps:.2f} FPS (Original)")

def run_external_batch_processing_with_fixed_padding(
    list_of_source_items: list, 
    yolo_model_instance: YOLO,
    fixed_patch_batch_size_for_model: int,
    crop_gen_params: dict, # Contains shape_x, shape_y, overlap_x, overlap_y
    common_mcdt_factory_config: dict, 
    global_resize_to_original_after_crop_projection: bool = True 
):
    all_crop_elements_global: list['CropElement'] = []
    all_crop_image_arrays_global: list[np.ndarray] = []
    
    crop_generator_factory = MakeCropsDetectThem(
        image=None, 
        model=yolo_model_instance,
        mode="generate_crops_only",
        shape_x=crop_gen_params['shape_x'],
        shape_y=crop_gen_params['shape_y'],
        overlap_x=crop_gen_params['overlap_x'],
        overlap_y=crop_gen_params['overlap_y'],
        resize_initial_size=global_resize_to_original_after_crop_projection, 
        **common_mcdt_factory_config 
    )

    max_crop_h, max_crop_w = 0, 0 
    # print(f"  Orchestrator: Generating crops for {len(list_of_source_items)} source items...")
    for image_arr, current_image_id_tuple in list_of_source_items:
        unique_processing_id = current_image_id_tuple 

        # print(f"    DEBUG Orchestrator: Generating crops for image ID {unique_processing_id}, shape {image_arr.shape}")
        crop_elements_for_img, crop_arrays_for_img = \
            crop_generator_factory.generate_crops_for_external_processing(
                image_to_crop=image_arr, # This should be the full resolution frame
                source_id=unique_processing_id 
            )
        # print(f"    DEBUG Orchestrator: Generated {len(crop_arrays_for_img)} crops for image ID {unique_processing_id}")
        
        all_crop_elements_global.extend(crop_elements_for_img)
        for arr in crop_arrays_for_img:
            if arr is not None and arr.size > 0:
                all_crop_image_arrays_global.append(arr)
                h, w = arr.shape[:2]
                if h > max_crop_h: max_crop_h = h
                if w > max_crop_w: max_crop_w = w
    
    if not all_crop_elements_global: 
        # print("  Orchestrator: No crops generated from any source items.")
        return {}

    pad_h = int(max_crop_h if max_crop_h > 0 else crop_gen_params.get('shape_y', 640))
    pad_w = int(max_crop_w if max_crop_w > 0 else crop_gen_params.get('shape_x', 640))
    pad_c = 3 
    pad_h = max(32, pad_h); pad_w = max(32, pad_w)
    padding_image = np.zeros((pad_h, pad_w, pad_c), dtype=np.uint8)
    # print(f"  Orchestrator: Using padding image of shape: {padding_image.shape}")


    # print(f"  Orchestrator: Performing inference on {len(all_crop_image_arrays_global)} crops, fixed batch size {fixed_patch_batch_size_for_model}...")
    num_actual_patches = len(all_crop_image_arrays_global)
    
    for i in range(0, num_actual_patches, fixed_patch_batch_size_for_model):
        actual_batch_arrays = all_crop_image_arrays_global[i : min(i + fixed_patch_batch_size_for_model, num_actual_patches)]
        actual_batch_elements = all_crop_elements_global[i : min(i + fixed_patch_batch_size_for_model, num_actual_patches)]
        
        num_in_actual_batch = len(actual_batch_arrays)
        model_input_batch_arrays = list(actual_batch_arrays)
        
        if num_in_actual_batch < fixed_patch_batch_size_for_model:
            num_to_pad = fixed_patch_batch_size_for_model - num_in_actual_batch
            for _ in range(num_to_pad): model_input_batch_arrays.append(padding_image.copy())
        
        if not model_input_batch_arrays: continue

        predictions_for_model_batch = MakeCropsDetectThem._calculate_batch_inference_static(
            model_input_batch_arrays, yolo_model_instance,
            imgsz=common_mcdt_factory_config.get('imgsz', CROP_HEIGHT), # Use crop size for inference
            conf=common_mcdt_factory_config.get('conf', BASE_MODEL_CONF_FOR_CROPS),
            iou=common_mcdt_factory_config.get('iou', 0.7),
            classes_list=common_mcdt_factory_config.get('classes_list', None),
            extra_args=common_mcdt_factory_config.get('inference_extra_args', None)
        )
        
        for pred_idx, pred_result in enumerate(predictions_for_model_batch):
            if pred_idx < num_in_actual_batch: 
                crop_elem = actual_batch_elements[pred_idx]
                MakeCropsDetectThem.populate_crop_element_with_prediction(
                    crop_elem, pred_result,
                    segment_mode=common_mcdt_factory_config.get('segment', False),
                    memory_optimize_mode=common_mcdt_factory_config.get('memory_optimize', True)
                )
                crop_elem.calculate_real_values()
                if global_resize_to_original_after_crop_projection: 
                    crop_elem.resize_results()
    
    # print("  Orchestrator: Grouping results by original image ID...")
    grouped_crop_elements = defaultdict(list)
    for ce in all_crop_elements_global:
        grouped_crop_elements[ce.source_image_unique_id].append(ce)
    
    return grouped_crop_elements

def extract_raw_detections_from_single_frame_group(
    list_of_crop_elements_for_frame: list['CropElement'], 
    conf_threshold_for_raw: float
):
    raw_boxes, raw_confidences, raw_classes = [], [], []
    if not list_of_crop_elements_for_frame: return raw_boxes, raw_confidences, raw_classes

    for crop_element in list_of_crop_elements_for_frame:
        if hasattr(crop_element, 'detected_conf') and crop_element.detected_conf and len(crop_element.detected_conf) > 0:
            for i in range(len(crop_element.detected_conf)):
                conf = float(crop_element.detected_conf[i])
                if conf >= conf_threshold_for_raw:
                    if i < len(crop_element.detected_xyxy_real) and \
                       i < len(crop_element.detected_cls):
                        raw_boxes.append(crop_element.detected_xyxy_real[i])
                        raw_confidences.append(conf)
                        raw_classes.append(int(crop_element.detected_cls[i]))
    return raw_boxes, raw_confidences, raw_classes

# Initialize Counters
frames_actually_processed_count = 0
total_orchestrator_time_ms = 0 
total_nms_combination_time_ms_agg = 0
total_raw_extraction_time_ms_agg = 0
total_crops_generated_agg = 0
total_raw_detections_from_crops_agg = 0 
total_nms_detections_agg = 0
RESIZE_RESULTS_TO_ORIGINAL_FOR_THIS_RUN = True

cap_for_originals = cv2.VideoCapture(VIDEO_PATH)
if not cap_for_originals.isOpened():
    raise IOError(f"Failed to open video {VIDEO_PATH} for reading original frames.")

reader_frame_source = None
if os.path.exists(VIDEO_PATH):
    print(f"PVR: Initializing for {VIDEO_PATH}, target resize (shorter/longer): {MODEL_INPUT_SIZE}")
    reader_frame_source = PyVideoReader(VIDEO_PATH, device='cpu',
                                        resize_shorter_side=MODEL_INPUT_SIZE,
                                        resize_longer_side=MODEL_INPUT_SIZE)
    num_pvr_segments = math.ceil(total_video_frames / PYVIDEOREADER_DECODE_SEGMENT_SIZE) if total_video_frames > 0 else 0
    segment_iterator = range(num_pvr_segments)
    print(f"PVR Source: Segments={num_pvr_segments}")
else:
    raise FileNotFoundError(f"CRITICAL ERROR: Video path {VIDEO_PATH} does not exist.")

current_processing_pass_id = 0 

for seg_idx in segment_iterator:
    if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed_count >= MAX_FRAMES_TO_PROCESS:
        break

    pvr_segment_start_raw_idx = seg_idx * PYVIDEOREADER_DECODE_SEGMENT_SIZE
    segment_end_raw_idx = min((seg_idx + 1) * PYVIDEOREADER_DECODE_SEGMENT_SIZE, total_video_frames)
    if pvr_segment_start_raw_idx >= segment_end_raw_idx: continue
    
    pvr_frames_rgb_segment = reader_frame_source.decode(start_frame=pvr_segment_start_raw_idx,
                                                        end_frame=segment_end_raw_idx)
    if pvr_frames_rgb_segment is None or pvr_frames_rgb_segment.shape[0] == 0: continue
    
    pvr_frames_bgr_segment = [cv2.cvtColor(rgb_frame, cv2.COLOR_RGB2BGR) for rgb_frame in pvr_frames_rgb_segment]
    del pvr_frames_rgb_segment; gc.collect()
    
    frames_for_orchestrator_in_segment = []
    original_frames_bgr_for_drawing_this_segment = {} 

    for frame_idx_in_pvr_seg, _ in enumerate(pvr_frames_bgr_segment): # Iterate by index
        current_original_frame_idx = pvr_segment_start_raw_idx + frame_idx_in_pvr_seg
        if current_original_frame_idx % FRAME_INTERVAL_CONFIG != 0: continue
        if MAX_FRAMES_TO_PROCESS is not None and \
           (frames_actually_processed_count + len(frames_for_orchestrator_in_segment)) >= MAX_FRAMES_TO_PROCESS and \
           len(frames_for_orchestrator_in_segment) > 0 : # Check before adding more than needed
            break 
        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed_count >= MAX_FRAMES_TO_PROCESS:
             break


        cap_for_originals.set(cv2.CAP_PROP_POS_FRAMES, current_original_frame_idx)
        ret_orig, original_bgr_direct_for_drawing = cap_for_originals.read()
        if not ret_orig or original_bgr_direct_for_drawing is None:
            # Fallback to PVR frame if direct read fails, PVR frame might be resized
            original_frames_bgr_for_drawing_this_segment[current_original_frame_idx] = pvr_frames_bgr_segment[frame_idx_in_pvr_seg].copy()
            image_for_cropping = pvr_frames_bgr_segment[frame_idx_in_pvr_seg].copy()
        else:
            original_frames_bgr_for_drawing_this_segment[current_original_frame_idx] = original_bgr_direct_for_drawing
            image_for_cropping = original_bgr_direct_for_drawing.copy() # Crop from original full-res frame

        frames_for_orchestrator_in_segment.append(
            (image_for_cropping, (current_original_frame_idx, current_processing_pass_id))
        )
    del pvr_frames_bgr_segment; gc.collect()

    if not frames_for_orchestrator_in_segment: continue

    orchestrator_start_time_seg = time.monotonic()
    results_grouped_by_frame_id_seg = run_external_batch_processing_with_fixed_padding(
        list_of_source_items=frames_for_orchestrator_in_segment,
        yolo_model_instance=model,
        fixed_patch_batch_size_for_model=FIXED_PATCH_BATCH_SIZE_FOR_MODEL,
        crop_gen_params=ORCH_CROP_PARAMS,
        common_mcdt_factory_config=ORCH_COMMON_MCDT_CONFIG,
        global_resize_to_original_after_crop_projection=RESIZE_RESULTS_TO_ORIGINAL_FOR_THIS_RUN 
    )
    orchestrator_time_ms_seg = (time.monotonic() - orchestrator_start_time_seg) * 1000
    total_orchestrator_time_ms += orchestrator_time_ms_seg

    for unique_proc_id_tuple, crop_elements_for_frame in results_grouped_by_frame_id_seg.items():
        original_frame_idx = unique_proc_id_tuple[0]

        if MAX_FRAMES_TO_PROCESS is not None and frames_actually_processed_count >= MAX_FRAMES_TO_PROCESS: break
        
        if not crop_elements_for_frame: continue
        total_crops_generated_agg += len(crop_elements_for_frame)

        original_frame_to_draw_on = original_frames_bgr_for_drawing_this_segment.get(original_frame_idx)
        if original_frame_to_draw_on is None: continue
        
        raw_extraction_start = time.monotonic()
        raw_boxes, raw_confidences, raw_classes = extract_raw_detections_from_single_frame_group(
            crop_elements_for_frame, BASE_MODEL_CONF_FOR_CROPS
        )
        total_raw_extraction_time_ms_agg += (time.monotonic() - raw_extraction_start) * 1000
        total_raw_detections_from_crops_agg += len(raw_boxes)
        
        print("raw_classes:",raw_classes)
        print("class_names_map:",class_names_map)
        raw_annotated_frame, _ = annotate_detections(
            original_frame_to_draw_on, raw_boxes, raw_confidences, raw_classes,
            class_names_map, (0, 0, 255), POSTPROCESS_CONF_THRESHOLD, ""
        )
        cv2.imwrite(os.path.join(OUTPUT_DIR_RAW_CROPS, f"frame_{original_frame_idx:06d}_raw_crops.jpg"), raw_annotated_frame)

        RESIZE_RESULTS_TO_ORIGINAL_FOR_THIS_RUN = True
        nms_combination_start = time.monotonic()
        ref_element = crop_elements_for_frame[0]
        img_ctx_nms = ref_element.source_image if RESIZE_RESULTS_TO_ORIGINAL_FOR_THIS_RUN else ref_element.source_image_resized
        
        mcdt_wrapper = MCDTLike(
            crops_list=crop_elements_for_frame, class_names=model.names,
            segment_flag=ORCH_COMMON_MCDT_CONFIG['segment'],
            memory_optimize_flag=ORCH_COMMON_MCDT_CONFIG['memory_optimize'],
            resize_initial_size_flag=RESIZE_RESULTS_TO_ORIGINAL_FOR_THIS_RUN, 
            image_ref=img_ctx_nms
        )
        combined_result = CombineDetections(element_crops=mcdt_wrapper, **NMS_CONFIG)
        total_nms_combination_time_ms_agg += (time.monotonic() - nms_combination_start) * 1000
        
        num_nms_dets_frame = len(combined_result.filtered_boxes)
        total_nms_detections_agg += num_nms_dets_frame

        nms_annotated_frame, _ = annotate_detections(
            original_frame_to_draw_on, combined_result.filtered_boxes, 
            combined_result.filtered_confidences, combined_result.filtered_classes_id,
            class_names_map, (0, 255, 0), POSTPROCESS_CONF_THRESHOLD, ""
        )
        cv2.imwrite(os.path.join(OUTPUT_DIR_NMS, f"frame_{original_frame_idx:06d}_nms.jpg"), nms_annotated_frame)
        
        frames_actually_processed_count += 1 # Increment after full processing of this frame
    
    current_processing_pass_id +=1
    del frames_for_orchestrator_in_segment, results_grouped_by_frame_id_seg, original_frames_bgr_for_drawing_this_segment
    gc.collect()

if reader_frame_source: del reader_frame_source
if cap_for_originals.isOpened(): cap_for_originals.release()
gc.collect()

# Final Summary (using aggregated counters)
print("\n--- Refactored Benchmark Summary ---")
if frames_actually_processed_count > 0:
    avg_orchestrator_ms = total_orchestrator_time_ms / frames_actually_processed_count
    avg_nms_ms_final = total_nms_combination_time_ms_agg / frames_actually_processed_count
    avg_raw_extract_ms = total_raw_extraction_time_ms_agg / frames_actually_processed_count
    avg_crops_per_proc_frame = total_crops_generated_agg / frames_actually_processed_count
    avg_raw_dets_per_proc_frame = total_raw_detections_from_crops_agg / frames_actually_processed_count
    avg_nms_dets_per_proc_frame = total_nms_detections_agg / frames_actually_processed_count
    
    print(f"**Processing Statistics:**")
    print(f"  Total Video Frames Source: {total_video_frames}")
    print(f"  Frames Processed: {frames_actually_processed_count}") # Changed from Orchestrator
    print(f"  Total Crops Generated: {total_crops_generated_agg} (avg: {avg_crops_per_proc_frame:.1f}/processed frame)")
    print(f"  Total Raw Detections (from crops, >=BASE_MODEL_CONF): {total_raw_detections_from_crops_agg} (avg: {avg_raw_dets_per_proc_frame:.1f}/processed frame)")
    print(f"  Total NMS Detections (after CombineDetections): {total_nms_detections_agg} (avg: {avg_nms_dets_per_proc_frame:.1f}/processed frame)")
    if total_raw_detections_from_crops_agg > 0:
        reduction = (total_raw_detections_from_crops_agg - total_nms_detections_agg) / total_raw_detections_from_crops_agg * 100
        print(f"  NMS Reduction Rate (vs raw from crops): {reduction:.1f}%")
    
    print(f"\n**Timing Performance (per processed frame):**")
    print(f"  Orchestrator (Cropping + Batched Inference): {avg_orchestrator_ms:.2f} ms/frame")
    print(f"  Raw Detections Extraction from Crops: {avg_raw_extract_ms:.2f} ms/frame")
    print(f"  NMS Combination: {avg_nms_ms_final:.2f} ms/frame")
    
    total_cycle_time_ms_per_frame = avg_orchestrator_ms + avg_raw_extract_ms + avg_nms_ms_final
    print(f"  Total Avg End-to-End Cycle Time (for processed frames): {total_cycle_time_ms_per_frame:.2f} ms/frame")
    if total_cycle_time_ms_per_frame > 0: 
        equivalent_fps_processed = 1000 / total_cycle_time_ms_per_frame
        print(f"  Equivalent End-to-End FPS (for processed frames): {equivalent_fps_processed:.2f}")
else:
    print("No frames were processed.")

raw_files_count_final = len(os.listdir(OUTPUT_DIR_RAW_CROPS)) if os.path.exists(OUTPUT_DIR_RAW_CROPS) else 0
nms_files_count_final = len(os.listdir(OUTPUT_DIR_NMS)) if os.path.exists(OUTPUT_DIR_NMS) else 0
print(f"\n**Output Files:**")
print(f"  Drawn raw crop detections: {raw_files_count_final} files in {OUTPUT_DIR_RAW_CROPS}")
print(f"  Drawn NMS filtered: {nms_files_count_final} files in {OUTPUT_DIR_NMS}")
print("Benchmark finished.")

  Crop Params: Crop HxW=640x640, Overlap=20%, Post-Conf=0.3
  Override: CONF_THRESHOLD_OVERRIDE (for base model on crops) set to 0.15
Crop Params: Crop HxW=640x640, Overlap=20%, Post-Conf(Draw)=0.3
  Effective CONF for base model on crops set to 0.15
NMS Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_ext_batch_20250622_2357\nms_filtered
Raw Crop Detections Output: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\benchmark_detection_outputs\5 min test - Large FOV_crops_ext_batch_20250622_2357\raw_crop_detections
Crop-based inference setup applied.
--------------------------------------------------

--- Starting Benchmark Processing (Refactored) ---
Video Properties: ~7500 frames, 25.00 FPS (Original)
PVR: Initializing for C:\Users\swest\Desktop\traffic-capturing\sample_videos\5 min test - Large FOV.mp4, target resize (shorter/longer): 640
PVR Source: Segments=150
Loadin

WARNING Metadata not found for 'model=C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\silent_contract_9082_dynamic.engine'
raw_classes: [2, 0, 2, 9, 9, 2, 9, 2, 2, 9, 2, 2, 2, 0, 9, 9, 9, 2, 9, 2, 9, 2, 12, 2, 6, 13, 9, 0, 8, 2, 7, 0, 2, 9, 9, 9, 2, 2, 2, 9, 2, 2, 9, 9, 9, 2, 9, 9, 2, 2, 0, 2, 2, 0, 2, 9, 2, 2, 7, 0, 2, 9, 7, 2, 2, 9, 0, 2, 0, 7]
class_names_map: {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}
annotation complete
annotation complete
raw_classes: [2, 2, 2, 9, 9, 9, 9, 2, 2, 9, 2, 9, 2, 2, 9, 2, 9, 2, 2, 9, 2, 2, 7, 73, 9, 2, 6, 13, 9, 2, 2, 2, 0, 7, 9, 9, 9, 9, 2, 2, 9, 9, 9, 9, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 9, 2, 2, 7, 0, 2, 2, 9, 9, 0, 2, 0, 0]
class_names_map: {0: 'Bus', 1: 'Cyclist', 2: 'Light Vehicle', 3: 'Long Truck', 4: 'Medium Truck', 5: 'Minibus Taxi', 6: 'Motorcycle', 7: 'Pedestrian', 8: 'Short Truck'}
annotation complete
anno